In [10]:
import pandas as pd
import requests
import time
from tqdm import tqdm

# No LM Studio, o modelo usado é aquele que está carregado no momento. 
# Você pode colocar um nome genérico ou o nome exato do arquivo.
modelos_locais = ["mistral-7b", "qwen-2.5-7b", "llama-3.1-8b", "gemma-3-4b"]

prompts_mitigacao = {
    "aware": (
        "When analyzing information, remain aware of confirmation bias — "
        "the tendency to favor data that supports prior beliefs while ignoring conflicting evidence. "
        "Actively seek out contradictory information and incorporate it into your reasoning. "
        "Strive for a balanced and evidence-based response. Then give your answer in a direct way."
    ),
    "autocritica_simulada": (
        "Answer the question first. Then, reflect on your reasoning: "
        "Was any key evidence overlooked? Could your response have been influenced by confirmation bias? "
        "Revise your answer and state your impressions in an direct way."
    ),
    "multiplas_fontes": (
        "Ensure your answer is informed by diverse sources, not just those that align with initial assumptions. "
        "Actively seek out credible perspectives that challenge your viewpoint to increase objectivity and accuracy. "
        "Give your answer in a direct way."
    ),
    "protocolo_verificacao": (
        "Before forming a conclusion, apply this verification protocol:\n"
        "(1) Search for disconfirming evidence.\n"
        "(2) Consider the strongest counterarguments.\n"
        "(3) Check for cherry-picked data.\n"
        "(4) Review sources that challenge your assumptions.\n"
        "(5) Identify what evidence would change your mind.\n"
        "(6) Give your answer in a direct way.\n"
    )
}

def montar_prompt(instrucao, pergunta):
    return f"{instrucao}\n\nQuestion: {pergunta}"

def consultar_lm_studio(prompt):
    # O endpoint padrão do LM Studio
    url = "http://localhost:1234/v1/chat/completions"
    
    body = {
        "model": "local-model", # O LM Studio ignora este campo e usa o modelo carregado
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.7
    }

    try:
        response = requests.post(url, json=body, timeout=120)
        response.raise_for_status()
        return response.json()['choices'][0]['message']['content']
    except Exception as e:
        return f"Erro na conexão local: {str(e)}"

def testar_mitigacao_local(arquivo_perguntas, nome_modelo_atual):
    perguntas_df = pd.read_csv(arquivo_perguntas)
    resultados = []

    # Como o LM Studio só roda um modelo por vez, processamos o modelo carregado
    for _, linha in tqdm(perguntas_df.iterrows(), total=perguntas_df.shape[0], desc=f"Testando {nome_modelo_atual}"):
        pergunta = linha["pergunta"]
        pergunta_id = linha["id"]

        linha_resultado = {
            "id": pergunta_id,
            "pergunta": pergunta,
            "modelo": nome_modelo_atual
        }

        for chave_prompt, instrucao in prompts_mitigacao.items():
            prompt_final = montar_prompt(instrucao, pergunta)
            resposta = consultar_lm_studio(prompt_final)
            linha_resultado[f"resposta_{chave_prompt}"] = resposta

        resultados.append(linha_resultado)

    df_resultado = pd.DataFrame(resultados)
    nome_arquivo = f"respostas_local_{nome_modelo_atual.replace('.', '_')}.csv"
    df_resultado.to_csv(nome_arquivo, index=False)
    print(f"\n✅ Resultados salvos em: {nome_arquivo}")

# EXECUÇÃO
# 1. Carregue o Mistral no LM Studio e mude o nome abaixo:
testar_mitigacao_local("perguntas_enviesadas.csv", "gemma-3-4b")

Testando gemma-3-4b: 100%|██████████| 52/52 [6:35:03<00:00, 455.83s/it]  


✅ Resultados salvos em: respostas_local_gemma-3-4b.csv
